In [1]:
import os, json, zipfile, itertools, math, copy
import numpy as np
from pathlib import Path
import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto, shape_inference


In [2]:
TASK_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
C, H, W = 10, 30, 30

In [3]:
def load_task(path):
    with open(path) as f:
        return json.load(f)

def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            v = g[r, c]
            if 0 <= v <= 9:
                t[0, v, r, c] = 1.0
    return t

def tensor_to_grid(t, out_h, out_w):
    arr = t[0]
    grid = []
    for r in range(out_h):
        row = []
        for c in range(out_w):
            col_vals = arr[:, r, c]
            if col_vals.max() < 0.5:
                row.append(0)
            else:
                row.append(int(np.argmax(col_vals)))
        grid.append(row)
    return grid

def get_output_size(pairs):
    sizes = set()
    for p in pairs:
        g = np.array(p["output"])
        sizes.add(g.shape)
    return sizes

def get_input_size(pairs):
    sizes = set()
    for p in pairs:
        g = np.array(p["input"])
        sizes.add(g.shape)
    return sizes

def make_identity_onnx():
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node("Identity", ["input"], ["output"])
    graph = oh.make_graph([node], "identity", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_const_onnx(const_output):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    const_val = const_output.astype(np.float32)
    const_tensor = onh.from_array(const_val, name="const_out")
    const_node = oh.make_node("Constant", [], ["const_out"], value=const_tensor)
    node = oh.make_node("Add", ["input", "const_out"], ["pre_out"])
    zero_tensor = onh.from_array(np.zeros((1, C, H, W), dtype=np.float32), name="zero_val")
    zero_node = oh.make_node("Constant", [], ["zero_val"], value=zero_tensor)
    mul_node = oh.make_node("Mul", ["input", "zero_val"], ["zeroed"])
    add_node = oh.make_node("Add", ["zeroed", "const_out"], ["output"])
    graph = oh.make_graph([const_node, zero_node, mul_node, add_node], "const_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def check_model_correct(model, pairs):
    try:
        import onnxruntime as ort
    except ImportError:
        os.system("pip install onnxruntime -q")
        import onnxruntime as ort
    try:
        buf = model.SerializeToString()
        sess = ort.InferenceSession(buf, providers=["CPUExecutionProvider"])
        for p in pairs:
            inp = grid_to_tensor(p["input"])
            out_g = np.array(p["output"])
            oh_, ow_ = out_g.shape
            pred = sess.run(None, {"input": inp})[0]
            pred_grid = tensor_to_grid(pred, oh_, ow_)
            if pred_grid != p["output"]:
                return False
        return True
    except Exception:
        return False

def analyze_transformation(pairs):
    info = {}
    in_sizes = [np.array(p["input"]).shape for p in pairs]
    out_sizes = [np.array(p["output"]).shape for p in pairs]
    info["in_sizes"] = in_sizes
    info["out_sizes"] = out_sizes
    info["same_size"] = all(i == o for i, o in zip(in_sizes, out_sizes))
    info["const_output"] = len(set(map(lambda p: str(p["output"]), pairs))) == 1

    if info["same_size"] and len(in_sizes) > 0:
        diffs = []
        for p in pairs:
            ig = np.array(p["input"])
            og = np.array(p["output"])
            diffs.append(np.sum(ig != og))
        info["avg_diff"] = np.mean(diffs)
        info["identity"] = all(d == 0 for d in diffs)
    else:
        info["avg_diff"] = None
        info["identity"] = False

    color_maps = []
    for p in pairs:
        ig = np.array(p["input"]).flatten()
        og = np.array(p["output"]).flatten()
        if ig.shape == og.shape:
            cmap = {}
            valid = True
            for a, b in zip(ig, og):
                if a in cmap and cmap[a] != b:
                    valid = False
                    break
                cmap[a] = b
            if valid:
                color_maps.append(cmap)
    if color_maps and len(color_maps) == len(pairs):
        common = color_maps[0]
        for cm in color_maps[1:]:
            if cm != common:
                common = None
                break
        info["color_map"] = common
    else:
        info["color_map"] = None

    return info

def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"])
            og = np.array(p["output"])
            k = angle // 90
            rotated = np.rot90(ig, k)
            if rotated.shape != og.shape or not np.array_equal(rotated, og):
                valid = False
                break
        if valid:
            return angle
    return None

def detect_flip(pairs):
    for axis, name in [(0, "vertical"), (1, "horizontal"), (-1, "both")]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"])
            og = np.array(p["output"])
            if axis == -1:
                flipped = np.flip(ig, axis=0)
                flipped = np.flip(flipped, axis=1)
            else:
                flipped = np.flip(ig, axis=axis)
            if not np.array_equal(flipped, og):
                valid = False
                break
        if valid:
            return name
    return None

def detect_transpose(pairs):
    for p in pairs:
        ig = np.array(p["input"])
        og = np.array(p["output"])
        if ig.T.shape != og.shape or not np.array_equal(ig.T, og):
            return False
    return True

def detect_crop(pairs):
    results = []
    for p in pairs:
        ig = np.array(p["input"])
        og = np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ > ih or ow_ > iw:
            return None
        for r in range(ih - oh_ + 1):
            for c in range(iw - ow_ + 1):
                if np.array_equal(ig[r:r+oh_, c:c+ow_], og):
                    results.append((r, c, oh_, ow_))
                    break
    if len(results) == len(pairs):
        offsets = [(r[0], r[1]) for r in results]
        if len(set(offsets)) == 1:
            r0, c0 = offsets[0]
            oh_ = results[0][2]
            ow_ = results[0][3]
            return (r0, c0, oh_, ow_)
    return None

def detect_tile(pairs):
    for p in pairs:
        ig = np.array(p["input"])
        og = np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ % ih != 0 or ow_ % iw != 0:
            return None
        tr = oh_ // ih
        tc = ow_ // iw
        tiled = np.tile(ig, (tr, tc))
        if not np.array_equal(tiled, og):
            return None
    return True

def detect_scale(pairs):
    for factor in [2, 3, 4, 5]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"])
            og = np.array(p["output"])
            ih, iw = ig.shape
            oh_, ow_ = og.shape
            if oh_ != ih * factor or ow_ != iw * factor:
                valid = False
                break
            scaled = np.repeat(np.repeat(ig, factor, axis=0), factor, axis=1)
            if not np.array_equal(scaled, og):
                valid = False
                break
        if valid:
            return factor
    return None

In [4]:
def make_rotation_onnx(angle):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = []
    k = angle // 90

    perm_90 = [0, 1, 3, 2]
    if k == 1:
        flip_w = np.zeros((1, C, H, W), dtype=np.float32)
        for c_ in range(C):
            for r in range(H):
                for col in range(W):
                    new_r = col
                    new_c = H - 1 - r
                    if new_r < H and new_c < W:
                        flip_w[0, c_, new_r, new_c] = 1.0

    weight_data = np.zeros((C, C, H, W, H, W), dtype=np.float32)

    if k == 2:
        weight = np.zeros((C, C, H, W), dtype=np.float32)
        for ch in range(C):
            for r in range(H):
                for col in range(W):
                    weight[ch, ch, H-1-r, W-1-col] = 1.0

        w_tensor = onh.from_array(weight.reshape(C*C, H*W).T.reshape(H*W, C, C), name="unused")
        gather_indices = np.array([H*W - 1 - i for i in range(H*W)], dtype=np.int64)

        inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"],
                                   ["inp_chw"])
        shape_chw = onh.from_array(np.array([1, C, H*W], dtype=np.int64), name="shape_chw")
        shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)

        gi_tensor = onh.from_array(gather_indices, name="gather_idx")
        gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)

        gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

        out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
        out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])

        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]

    elif k == 1:
        gather_indices = np.zeros(H*W, dtype=np.int64)
        for r in range(H):
            for col in range(W):
                new_r = col
                new_c = H - 1 - r
                if new_r < H and new_c < W:
                    gather_indices[new_r * W + new_c] = r * W + col

        shape_chw = onh.from_array(np.array([1, C, H*W], dtype=np.int64), name="shape_chw")
        shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
        inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

        gi_tensor = onh.from_array(gather_indices, name="gather_idx")
        gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
        gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

        out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
        out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])

        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]

    elif k == 3:
        gather_indices = np.zeros(H*W, dtype=np.int64)
        for r in range(H):
            for col in range(W):
                new_r = W - 1 - col
                new_c = r
                if new_r < H and new_c < W:
                    gather_indices[new_r * W + new_c] = r * W + col

        shape_chw = onh.from_array(np.array([1, C, H*W], dtype=np.int64), name="shape_chw")
        shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
        inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

        gi_tensor = onh.from_array(gather_indices, name="gather_idx")
        gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
        gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

        out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
        out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])

        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]

    graph = oh.make_graph(nodes, "rotation_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_flip_onnx(direction):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    if direction == "vertical":
        gather_indices = np.zeros(H*W, dtype=np.int64)
        for r in range(H):
            for col in range(W):
                gather_indices[r * W + col] = (H - 1 - r) * W + col
    elif direction == "horizontal":
        gather_indices = np.zeros(H*W, dtype=np.int64)
        for r in range(H):
            for col in range(W):
                gather_indices[r * W + col] = r * W + (W - 1 - col)
    else:
        gather_indices = np.zeros(H*W, dtype=np.int64)
        for r in range(H):
            for col in range(W):
                gather_indices[r * W + col] = (H - 1 - r) * W + (W - 1 - col)

    shape_chw = onh.from_array(np.array([1, C, H*W], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

    gi_tensor = onh.from_array(gather_indices, name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])

    nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]
    graph = oh.make_graph(nodes, "flip_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_color_remap_onnx(color_map):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    for src, dst in color_map.items():
        if 0 <= src < C and 0 <= dst < C:
            weight[dst, src, 0, 0] = 1.0

    covered_dsts = set(color_map.values())
    for ch in range(C):
        if ch not in covered_dsts:
            found = False
            for src, dst in color_map.items():
                if dst == ch:
                    found = True
                    break
            if not found and ch not in color_map:
                weight[ch, ch, 0, 0] = 1.0

    w_tensor = onh.from_array(weight, name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                             kernel_shape=[1, 1], pads=[0, 0, 0, 0])

    nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, "color_remap", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_permute_pixels_onnx(gather_indices):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    shape_chw = onh.from_array(np.array([1, C, H*W], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

    gi_tensor = onh.from_array(gather_indices.astype(np.int64), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])

    nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]
    graph = oh.make_graph(nodes, "permute_pixels", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def detect_pixel_permutation(pairs):
    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None
    if not all(np.array(p["input"]).shape == (H, W) for p in pairs):
        return None

    first_pair = pairs[0]
    ig = np.array(first_pair["input"])
    og = np.array(first_pair["output"])

    candidate = np.full(H*W, -1, dtype=np.int64)
    src_flat = ig.flatten()
    dst_flat = og.flatten()

    assigned = {}
    for dst_idx in range(H*W):
        dst_val = dst_flat[dst_idx]
        matches = np.where(src_flat == dst_val)[0]
        if len(matches) == 1:
            assigned[dst_idx] = matches[0]
        else:
            return None

    gather_indices = np.array([assigned.get(i, i) for i in range(H*W)], dtype=np.int64)

    for p in pairs[1:]:
        ig2 = np.array(p["input"]).flatten()
        og2 = np.array(p["output"]).flatten()
        predicted = ig2[gather_indices]
        if not np.array_equal(predicted, og2):
            return None

    return gather_indices

def detect_and_make_linear_conv(pairs):
    if not all(np.array(p["input"]).shape == (H, W) and
               np.array(p["output"]).shape == (H, W) for p in pairs):
        return None

    inp_tensors = [grid_to_tensor(p["input"]) for p in pairs]
    out_tensors = [grid_to_tensor(p["output"]) for p in pairs]

    X_list = [t.reshape(-1) for t in inp_tensors]
    Y_list = [t.reshape(-1) for t in out_tensors]

    X_mat = np.stack(X_list)
    Y_mat = np.stack(Y_list)

    try:
        W_sol, res, rank, sv = np.linalg.lstsq(X_mat, Y_mat, rcond=None)
        Y_pred = X_mat @ W_sol
        if np.max(np.abs(Y_pred - Y_mat)) > 0.01:
            return None

        W_sol = W_sol.T.reshape(C, H, W, C, H, W)
        return W_sol
    except Exception:
        return None

In [5]:
def build_output_from_input_pattern(pairs):
    results = []
    for p in pairs:
        ig = np.array(p["input"])
        og = np.array(p["output"])
        results.append((ig, og))
    return results

def detect_selection_crop(pairs):
    results = []
    for p in pairs:
        ig = np.array(p["input"])
        og = np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ >= ih and ow_ >= iw:
            return None
        found = False
        for r0 in range(ih - oh_ + 1):
            for c0 in range(iw - ow_ + 1):
                region = ig[r0:r0+oh_, c0:c0+ow_]
                if np.array_equal(region, og):
                    results.append((r0, c0))
                    found = True
                    break
            if found:
                break
        if not found:
            return None
    if len(set(results)) == 1:
        r0, c0 = results[0]
        oh_ = np.array(pairs[0]["output"]).shape[0]
        ow_ = np.array(pairs[0]["output"]).shape[1]
        return r0, c0, oh_, ow_
    return None

def make_crop_onnx(r0, c0, oh_, ow_):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    gather_indices = np.zeros(H*W, dtype=np.int64)
    for r in range(H):
        for c_ in range(W):
            if r < oh_ and c_ < ow_:
                src_r = r0 + r
                src_c = c0 + c_
                if src_r < H and src_c < W:
                    gather_indices[r * W + c_] = src_r * W + src_c
                else:
                    gather_indices[r * W + c_] = 0
            else:
                gather_indices[r * W + c_] = 0

    shape_chw = onh.from_array(np.array([1, C, H*W], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices, name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

    if oh_ < H or ow_ < W:
        mask = np.zeros((1, C, H, W), dtype=np.float32)
        mask[0, :, :oh_, :ow_] = 1.0
        mask_tensor = onh.from_array(mask, name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
        out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"])
        mul_node = oh.make_node("Mul", ["gathered_4d", "mask"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, mask_node, mul_node]
    else:
        out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
        out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]

    graph = oh.make_graph(nodes, "crop_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def try_learned_conv(pairs, kernel_size=1):
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        return None

    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None

    inp_tensors = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out_tensors = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=False)
    nn.init.zeros_(conv.weight)

    optimizer = torch.optim.Adam(conv.parameters(), lr=0.01)
    best_loss = float("inf")
    best_state = None

    for step in range(500):
        optimizer.zero_grad()
        pred = conv(inp_tensors)
        loss = nn.functional.mse_loss(pred, out_tensors)
        loss.backward()
        optimizer.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = copy.deepcopy(conv.state_dict())

    if best_loss > 0.1:
        return None

    conv.load_state_dict(best_state)
    weight_np = conv.weight.detach().numpy()

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight_np.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                             kernel_shape=[kernel_size, kernel_size],
                             pads=[pad, pad, pad, pad])
    nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, "learned_conv", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def solve_task(task_data, task_name=""):
    all_pairs = task_data.get("train", []) + task_data.get("test", [])
    arc_gen = task_data.get("arc-gen", [])
    if arc_gen:
        all_pairs = all_pairs + arc_gen[:min(5, len(arc_gen))]

    train_pairs = task_data.get("train", [])
    if not train_pairs:
        return make_identity_onnx()

    info = analyze_transformation(train_pairs)

    if info.get("identity"):
        model = make_identity_onnx()
        if check_model_correct(model, all_pairs):
            return model

    if info.get("color_map") and info["same_size"]:
        model = make_color_remap_onnx(info["color_map"])
        if check_model_correct(model, all_pairs):
            return model

    rot = detect_rotation(train_pairs)
    if rot is not None:
        model = make_rotation_onnx(rot)
        if check_model_correct(model, all_pairs):
            return model

    flip = detect_flip(train_pairs)
    if flip is not None:
        model = make_flip_onnx(flip)
        if check_model_correct(model, all_pairs):
            return model

    if detect_transpose(train_pairs):
        gather_indices = np.zeros(H*W, dtype=np.int64)
        for r in range(H):
            for col in range(W):
                new_r = col
                new_c = r
                if new_r < H and new_c < W:
                    gather_indices[new_r * W + new_c] = r * W + col
        model = make_permute_pixels_onnx(gather_indices)
        if check_model_correct(model, all_pairs):
            return model

    crop = detect_selection_crop(train_pairs)
    if crop is not None:
        r0, c0, oh_, ow_ = crop
        model = make_crop_onnx(r0, c0, oh_, ow_)
        if check_model_correct(model, all_pairs):
            return model

    perm = detect_pixel_permutation(train_pairs)
    if perm is not None:
        model = make_permute_pixels_onnx(perm)
        if check_model_correct(model, all_pairs):
            return model

    for ks in [1, 3]:
        model = try_learned_conv(train_pairs, kernel_size=ks)
        if model is not None and check_model_correct(model, all_pairs):
            return model

    if info.get("const_output"):
        const_out = grid_to_tensor(train_pairs[0]["output"])
        model = make_const_onnx(const_out)
        if check_model_correct(model, all_pairs):
            return model

    try_general = try_general_linear(train_pairs, all_pairs)
    if try_general is not None:
        return try_general

    return None

def try_general_linear(train_pairs, all_pairs):
    if not all(np.array(p["input"]).shape == (H, W) and
               np.array(p["output"]).shape == (H, W) for p in train_pairs):
        return None

    inp_tensors = np.stack([grid_to_tensor(p["input"])[0] for p in train_pairs])
    out_tensors = np.stack([grid_to_tensor(p["output"])[0] for p in train_pairs])

    n, c, h, w = inp_tensors.shape
    X_mat = inp_tensors.reshape(n, -1)
    Y_mat = out_tensors.reshape(n, -1)

    try:
        W_sol, _, _, _ = np.linalg.lstsq(X_mat, Y_mat, rcond=None)
        Y_pred = X_mat @ W_sol
        err = np.max(np.abs(Y_pred.round() - Y_mat))
        if err > 0.1:
            return None
    except Exception:
        return None

    W_np = W_sol.T.astype(np.float32).reshape(c*h*w, c*h*w)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    flat_shape = onh.from_array(np.array([1, C*H*W], dtype=np.int64), name="flat_shape")
    flat_shape_node = oh.make_node("Constant", [], ["flat_shape"], value=flat_shape)
    reshape_in = oh.make_node("Reshape", ["input", "flat_shape"], ["inp_flat"])

    w_tensor = onh.from_array(W_np, name="gemm_weight")
    w_node = oh.make_node("Constant", [], ["gemm_weight"], value=w_tensor)
    gemm_node = oh.make_node("Gemm", ["inp_flat", "gemm_weight"], ["out_flat"],
                             transB=0, alpha=1.0, beta=0.0)

    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_out = oh.make_node("Reshape", ["out_flat", "out_shape"], ["output"])

    nodes = [flat_shape_node, reshape_in, w_node, gemm_node, out_shape_node, reshape_out]
    graph = oh.make_graph(nodes, "linear_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8

    if check_model_correct(model, all_pairs):
        return model
    return None

def save_model(model, path):
    try:
        onnx.checker.check_model(model)
    except Exception as e:
        print(f"  Warning: model check failed: {e}")
    with open(path, "wb") as f:
        f.write(model.SerializeToString())

def find_task_files():
    candidates = [
        TASK_DIR,
        Path("/kaggle/input"),
        Path("."),
    ]
    for d in candidates:
        files = list(d.glob("task*.json")) if d.exists() else []
        if files:
            return sorted(files)
    return []


In [6]:
task_files = find_task_files()
if not task_files:
    print("No task files found. Searching recursively...")
    for root, dirs, files in os.walk("/kaggle/input" if Path("/kaggle/input").exists() else "."):
        for fname in files:
            if fname.startswith("task") and fname.endswith(".json"):
                task_files.append(Path(root) / fname)
    task_files = sorted(task_files)

if not task_files:
    print("ERROR: No task files found.")

In [7]:
print(f"Found {len(task_files)} task files")

Found 400 task files


In [8]:
solved = []
failed = []
onnx_files = []

for task_path in task_files:
    task_name = task_path.stem
    task_num = task_name.replace("task", "")

    try:
        task_data = load_task(task_path)
    except Exception as e:
        print(f"  {task_name}: Failed to load - {e}")
        failed.append(task_name)
        continue

    print(f"Processing {task_name}...", end=" ", flush=True)

    try:
        model = solve_task(task_data, task_name)
    except Exception as e:
        print(f"ERROR: {e}")
        failed.append(task_name)
        continue

    if model is None:
        print("No solution found")
        failed.append(task_name)
        continue

    out_path = OUTPUT_DIR / f"task{task_num}.onnx"
    try:
        save_model(model, out_path)
        onnx_files.append(out_path)
        print(f"Solved -> {out_path.name}")
        solved.append(task_name)
    except Exception as e:
        print(f"Save error: {e}")
        failed.append(task_name)

print(f"\nSummary: {len(solved)} solved, {len(failed)} failed out of {len(task_files)} total")

Processing task001... No solution found
Processing task002...    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 70.1 MB/s eta 0:00:00
No solution found
Processing task003... No solution found
Processing task004... No solution found
Processing task005... No solution found
Processing task006... No solution found
Processing task007... No solution found
Processing task008... No solution found
Processing task009... No solution found
Processing task010... No solution found
Processing task011... No solution found
Processing task012... No solution found
Processing task013... No solution found
Processing task014... No solution found
Processing task015... Solved -> task015.onnx
Processing task016... Solved -> task016.onnx
Processing task017... No solution found
Processing task018... No solution found
Processing task019... No solution found
Processing task020... No solution found
Processing task021... No solution found
Processing task022... No solution found
Processing task023... No soluti

In [9]:
if failed:
    print(f"Failed tasks: {failed[:20]}{'...' if len(failed) > 20 else ''}")

if onnx_files:
    zip_path = OUTPUT_DIR / "submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in onnx_files:
            zf.write(f, f.name)
    print(f"\nSubmission zip created: {zip_path}")
    print(f"Zip size: {zip_path.stat().st_size / 1024:.1f} KB")
else:
    print("No ONNX files to package.")


Failed tasks: ['task001', 'task002', 'task003', 'task004', 'task005', 'task006', 'task007', 'task008', 'task009', 'task010', 'task011', 'task012', 'task013', 'task014', 'task017', 'task018', 'task019', 'task020', 'task021', 'task022']...

Submission zip created: /kaggle/working/submission.zip
Zip size: 18.1 KB
